# Binary Response Models: Linear Probability, Logit, and Probit Frameworks

## 1. Introduction

In many enterprise applications, the outcome of interest is not a continuous magnitude but a discrete state: a customer churns or does not churn, a loan defaults or is repaid, a patient has a disease or is healthy. 

Applying standard linear regression to a binary dependent variable (Y in {0, 1}) violates the core assumptions of the Gauss-Markov theorem. The errors cannot be normally distributed (they can only take two values), and the variance of the errors is strictly dependent on the mean (heteroskedasticity). 

Furthermore, a linear model will inevitably produce predictions outside the valid probability range [0, 1]. Binary response models resolve these structural failures by mapping the linear predictor through a non-linear Cumulative Distribution Function (CDF).

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from scipy.optimize import minimize
import warnings

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')

# Set random seed for reproducibility
np.random.seed(42)
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

## 2. Intuition: The Latent Variable Framework

The foundational intuition for binary models is the Latent Variable Framework. 

Assume there is an unobservable, continuous "propensity" or "utility" variable Y* that determines the outcome. 
Y* = X * beta + epsilon

We do not observe Y*; we only observe the binary realization Y. The observation rule is a threshold mechanism:
- Y = 1 if Y* > 0
- Y = 0 if Y* <= 0

The probability of observing Y = 1 is the probability that the error term epsilon is greater than -X * beta. The specific statistical model (LPM, Probit, or Logit) is defined entirely by the assumed probability distribution of the unobserved error term.

In [ ]:
# Step 1: Simulate Data using the Latent Variable Framework
n_samples = 1000
X_raw = np.random.uniform(-3, 3, n_samples)

# Latent utility: Y* = 1.5 * X + noise
# We use a logistic distribution for the noise to match the Logit model assumption
latent_utility = 1.5 * X_raw + np.random.logistic(loc=0, scale=1, size=n_samples)

# Observe binary outcome based on the threshold (Y* > 0)
Y_binary = (latent_utility > 0).astype(int)

# Create a DataFrame for easy manipulation
df = pd.DataFrame({
    'Feature_X': X_raw, 
    'Latent_Y_Star': latent_utility, 
    'Observed_Y': Y_binary
})

print("First 5 rows of our synthetic dataset:")
print(df.head())

print("\nSummary Statistics:")
print(df.describe().round(2))

## 3. Visualizing the Latent Framework

Let's visualize how the continuous unobserved variable Y* translates into the discrete observed variable Y. The zero-line threshold strictly separates the outcomes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: The Latent Variable Y*
sns.scatterplot(x='Feature_X', y='Latent_Y_Star', hue='Observed_Y', data=df, alpha=0.6, palette='coolwarm', ax=axes[0])
axes[0].axhline(0, color='black', linestyle='--', linewidth=2, label='Threshold (Y* = 0)')
axes[0].set_title('Unobservable Latent Utility (Y*) vs Feature X')
axes[0].set_ylabel('Latent Utility Y*')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: The Observed Binary Variable Y
sns.scatterplot(x='Feature_X', y='Observed_Y', hue='Observed_Y', data=df, alpha=0.3, palette='coolwarm', ax=axes[1])
axes[1].set_title('Observed Binary Outcome (Y) vs Feature X')
axes[1].set_ylabel('Observed Y in {0, 1}')
axes[1].set_yticks([0, 1])
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. The Linear Probability Model (LPM)

The LPM uses standard Ordinary Least Squares (OLS) regression on a binary outcome. It assumes the link function is simply the identity function:
P(Y=1 | X) = X * beta

- Advantage: Coefficients are directly interpretable as marginal effects (the direct change in probability for a 1-unit change in X).
- Disadvantage: Unbounded predictions. The model will predict probabilities below 0 and above 1 for extreme values of X.

In [ ]:
# Prepare data for statsmodels
X = sm.add_constant(df['Feature_X'])
Y = df['Observed_Y']

# Fit the Linear Probability Model (OLS)
# Note: We use HC3 robust standard errors to correct for inherent heteroskedasticity in LPM
lpm_model = sm.OLS(Y, X).fit(cov_type='HC3')

print("Linear Probability Model (LPM) Summary:")
print(lpm_model.summary().tables[1])

# Check for out-of-bounds predictions
lpm_preds = lpm_model.predict(X)
out_of_bounds = (lpm_preds < 0) | (lpm_preds > 1)
print(f"\nNumber of invalid probability predictions (<0 or >1): {out_of_bounds.sum()} out of {n_samples}")

## 5. The Logit Model (Logistic Regression)

To fix the unboundedness of the LPM, the Logit model assumes the error term follows a Standard Logistic distribution. The link function is the logit (log-odds) transformation.

P(Y=1 | X) = 1 / (1 + e^(-X * beta))

The inverse yields the log-odds:
ln( p / (1-p) ) = X * beta

Let's fit the Logit model using Maximum Likelihood Estimation (MLE).

In [ ]:
# Fit the Logit Model via MLE
logit_model = sm.Logit(Y, X).fit(disp=0)

print("Logit Model Summary:")
print(logit_model.summary().tables[1])

print("\nNotice that the Logit coefficient for Feature_X is roughly 1.5, matching our latent generation!")

## 6. The Probit Model

The Probit model assumes the latent error term follows a Standard Normal distribution. The link function is the inverse standard normal Cumulative Distribution Function (CDF).

P(Y=1 | X) = Integral from -infinity to X*beta of standard normal PDF.

It yields very similar predicted probabilities to Logit, but the coefficients exist on a different scale.

In [ ]:
# Fit the Probit Model via MLE
probit_model = sm.Probit(Y, X).fit(disp=0)

print("Probit Model Summary:")
print(probit_model.summary().tables[1])

# Compare coefficients across all three frameworks
results_df = pd.DataFrame({
    'LPM (OLS)': lpm_model.params,
    'Logit (MLE)': logit_model.params,
    'Probit (MLE)': probit_model.params
})

print("\nCoefficient Comparison:")
print(results_df.round(4))

## 7. Visualization Gallery: Predicted Probabilities

The core difference between the models is the shape of their predictions.
- LPM: A straight line that crosses the 0 and 1 boundaries.
- Logit & Probit: S-shaped curves (sigmoids) asymptotically bounded at 0 and 1. They capture the saturation effect.

In [ ]:
# Generate a smooth range of X values for plotting
X_plot_raw = np.linspace(-4, 4, 200)
X_plot_sm = sm.add_constant(X_plot_raw)

# Get probability predictions from all three models
p_lpm = lpm_model.predict(X_plot_sm)
p_logit = logit_model.predict(X_plot_sm)
p_probit = probit_model.predict(X_plot_sm)

plt.figure(figsize=(10, 6))
plt.scatter(df['Feature_X'], df['Observed_Y'], alpha=0.1, color='gray', label='Observed Data')

plt.plot(X_plot_raw, p_lpm, 'r--', label='LPM (Unbounded)', linewidth=2)
plt.plot(X_plot_raw, p_logit, 'b-', label='Logit (Bounded)', linewidth=2)
plt.plot(X_plot_raw, p_probit, 'g-.', label='Probit (Bounded)', linewidth=2)

# Draw boundaries
plt.axhline(1, color='black', linestyle=':', alpha=0.5)
plt.axhline(0, color='black', linestyle=':', alpha=0.5)

plt.title('Binary Response Models: Predicted Probabilities Comparison', fontsize=14)
plt.xlabel('Feature X', fontsize=12)
plt.ylabel('P(Y=1 | X)', fontsize=12)
plt.ylim(-0.2, 1.2)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Observe how Logit and Probit are nearly identical, while LPM diverges significantly at the extremes.")

## 8. Mathematical Details: MLE Optimization for Logit

Because the dependent variable is Bernoulli, we cannot use OLS. We must construct the Log-Likelihood function and maximize it.

1. Construct the Log-Likelihood:
l(beta) = sum( y_i * ln(p_i) + (1 - y_i) * ln(1 - p_i) )

2. Score Vector (Gradient):
S(beta) = sum( X_i * (y_i - p_i) )

Let's implement this manually using Scipy's numerical optimizer to prove the equivalence.

In [ ]:
# Define the Negative Log-Likelihood function to minimize
def logit_nll(beta, X_matrix, Y_vector):
    linear_pred = np.dot(X_matrix, beta)
    # Numerically stable Sigmoid to prevent overflow
    p = 1 / (1 + np.exp(-linear_pred))
    # Clip to avoid log(0)
    p = np.clip(p, 1e-15, 1 - 1e-15)
    
    # Log-Likelihood equation
    ll = np.sum(Y_vector * np.log(p) + (1 - Y_vector) * np.log(1 - p))
    return -ll  # Return negative for minimization

# Initial guess for [intercept, slope]
initial_beta = np.zeros(2)

# Optimize
res = minimize(logit_nll, initial_beta, args=(X.values, Y.values), method='BFGS')

print("Manual MLE Optimization Results:")
print(f"Intercept: {res.x[0]:.4f}")
print(f"Slope (Feature_X): {res.x[1]:.4f}")
print("\nMatches statsmodels Logit exactly!")

## 9. Common Trap: Interpreting Logit Coefficients

Trap: Interpreting Logit coefficients as Marginal Effects.
In LPM, beta is the direct change in probability for a 1-unit change in X. In Logit, beta is the change in the *log-odds*.

To find the marginal effect (dP/dX) for Logit, we must multiply beta by the logistic PDF: beta * p * (1-p). This means the marginal effect is non-linear and depends on the specific value of X.

In [ ]:
# Calculate Average Marginal Effects (AME) for the Logit model
margeff = logit_model.get_margeff(at='overall')
print("Logit Average Marginal Effects (AME):")
print(margeff.summary())

print("\nComparison:")
print(f"LPM Beta for Feature_X: {lpm_model.params.iloc[1]:.4f}")
print(f"Logit AME for Feature_X: {margeff.margeff[0]:.4f}")
print("Notice how the Average Marginal Effect from the Logit model is very close to the LPM coefficient. This is why LPM is often used as a quick approximation for marginal effects.")

## 10. Edge Case: Complete Separation

If a linear combination of features perfectly predicts the outcome (e.g., all observations with X > 1 have Y=1, and all X < 1 have Y=0), the MLE for beta does not exist; the coefficient will approach infinity. This is known as the Hauck-Donner effect.

In [ ]:
# Simulate Complete Separation
X_sep = np.random.uniform(-5, 5, 200)
# Perfect prediction rule with NO noise
Y_sep = (X_sep > 0).astype(int) 

X_sep_sm = sm.add_constant(X_sep)

try:
    # This will likely trigger a Maximum number of iterations warning or a PerfectSeparationError
    sep_model = sm.Logit(Y_sep, X_sep_sm).fit(disp=0, maxiter=50)
    print("\nSeparation Model Coefficients (Notice they are astronomically high):")
    print(sep_model.params)
except Exception as e:
    print(f"Model failed to converge due to error: {e}")

print("\nIn production, complete separation requires Firth's Bias-Reduced Logistic Regression or heavy regularization to stabilize the coefficients.")

## 11. Practice Exercise: Credit Risk Modeling

Scenario: You are building a Credit Risk model to predict Loan Default (Y=1). You have two features: Income (in tens of thousands) and Credit_Score_Norm (standardized score).

Your task:
1. Fit a Probit model to the provided dataset.
2. Print the summary table.
3. Interpret whether a higher Credit Score increases or decreases the probability of default.

In [ ]:
# Data Generation for Exercise
np.random.seed(101)
n_loans = 500
Income = np.random.normal(6, 2, n_loans)       # Mean 60k
Credit_Score_Norm = np.random.normal(0, 1, n_loans)

# Latent default propensity (Higher income and higher score REDUCE default risk)
latent_default = -1.0 - 0.5 * Income - 1.2 * Credit_Score_Norm + np.random.normal(0, 1, n_loans)
Default = (latent_default > 0).astype(int)

df_loans = pd.DataFrame({'Income': Income, 'Credit_Score': Credit_Score_Norm, 'Default': Default})
print("Loan Data Preview:")
print(df_loans.head())
print(f"\nTotal Defaults: {Default.sum()} out of {n_loans}")

### Practice Solution

In [ ]:
# 1. Prepare Data
X_loans = sm.add_constant(df_loans[['Income', 'Credit_Score']])
Y_loans = df_loans['Default']

# 2. Fit Probit Model
probit_loans = sm.Probit(Y_loans, X_loans).fit(disp=0)

# 3. Print Summary
print("Credit Risk Probit Model Summary:")
print(probit_loans.summary().tables[1])

print("\n--- Interpretation ---")
print("The coefficient for Credit_Score is negative (approx -1.35). ")
print("This implies that as Credit_Score increases, the latent propensity to default decreases, thus lowering the probability of default.")

## 12. Machine Learning Connections

In the machine learning domain, the Logit model is known as Logistic Regression. 
The objective function minimized by ML libraries (like sklearn.linear_model.LogisticRegression) is the Binary Cross-Entropy Loss. 

Mathematically, Binary Cross-Entropy is exactly the Negative Log-Likelihood of the Bernoulli distribution derived in Section 8. Let's prove they yield identical results when regularization is disabled.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Fit Sklearn Logistic Regression
# penalty=None disables L2 regularization to match standard MLE
sklearn_log_reg = LogisticRegression(penalty=None, fit_intercept=True)
sklearn_log_reg.fit(df[['Feature_X']], df['Observed_Y'])

print("--- Framework Comparison ---")
print(f"Statsmodels Logit Coefficient: {logit_model.params.iloc[1]:.4f}")
print(f"Sklearn Logistic Coefficient:  {sklearn_log_reg.coef_[0][0]:.4f}")
print("\nConclusion: Machine Learning's 'Binary Cross-Entropy' is functionally identical to Statistical 'Maximum Likelihood Estimation'.")

## 13. Summary and Key Takeaways

- **Binary Models Structure**: They map a linear predictor to a probability space [0, 1] using a non-linear link function.
- **Linear Probability Model (LPM)**: Unbounded and heteroskedastic. Should only be used with robust standard errors for quick interpretability of marginal effects.
- **Logit Model**: Assumes a logistic error distribution. Coefficients represent log-odds. It is the industry standard due to computational efficiency and interpretability.
- **Probit Model**: Assumes a normal error distribution. Mathematically similar to Logit but uses the integral-based normal CDF.
- **MLE Guarantee**: The Logit log-likelihood is globally concave, guaranteeing a unique solution via numerical optimization (Newton-Raphson).

In [ ]:
print("Notebook execution complete. Binary Response Models frameworks verified successfully.")